# Confusion matrix for the solar event identification checkpoint

This notebook loads a trained checkpoint from `3_finetune_template_1D.py`, runs it over
the validation set, and reports the confusion matrix, precision/recall/F1, and accuracy —
not just `val_loss`.

**Why this matters here:** the label (`Solar_Event_Occurrence`) is imbalanced
(~62.6% 0 / 37.4% 1 in the full catalog). A model that always predicts 0 already scores
~62.5% accuracy and a binary cross-entropy loss close to what this notebook was written to
check for. The confusion matrix (and precision/recall *per class*) is what actually shows
whether the model learned something beyond the class prior, or is just predicting the
majority class every time.

**Before trusting these numbers:** make sure the checkpoint you point at was trained with
`data.max_samples` set to `null` (or a large number) in the config — evaluating a model
trained on a handful of samples will not tell you anything reliable, no matter how the
confusion matrix looks.

In [1]:
import os

# Must be set BEFORE torch is imported — see 3_finetune_template_1D.py for why.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
sys.path.append("../../")  # repo root, so workshop_infrastructure/ and downstream_apps/ are importable

import importlib

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

from downstream_apps.solar_event.configs import solar_event_config
from workshop_infrastructure.assets import ensure_assets
from workshop_infrastructure.utils import build_scalers

# "3_finetune_template_1D" starts with a digit, so it is not a valid identifier and
# `from ... import ...` cannot name it directly (this is also why the script itself is
# invoked with `python -m ...` rather than a plain import). importlib works around that,
# and reusing build_datasets()/build_model() means evaluation can never silently drift
# from how the checkpoint was actually trained.
finetune_script = importlib.import_module("downstream_apps.solar_event.3_finetune_template_1D")
build_datasets = finetune_script.build_datasets
build_model = finetune_script.build_model

torch.set_float32_matmul_precision("medium")

## Load config and point at the checkpoint to evaluate

`max_samples` is forced to `None` here regardless of what the YAML says, so evaluation
always runs over the *full* validation set — the training config may well have a small
`max_samples` for quick iteration, but that cap should never apply to evaluation.

Set `CKPT_PATH` to the checkpoint you want to evaluate, and `TRAIN_BASELINE` to match how
it was trained (`--train_baseline` was passed, or not) — the architecture has to match the
checkpoint's state dict exactly.

In [2]:
from pathlib import Path

CONFIG_PATH = "./configs/config_script.yaml"
TRAIN_BASELINE = False  # set True if the checkpoint was trained with --train_baseline

cfg = solar_event_config(CONFIG_PATH)
cfg.data.max_samples = None  # evaluate on the full validation set, regardless of the YAML
print(f"Loaded config for job: {cfg.job_id}")

# output.ckpt_dir is resolved relative to the process's CURRENT WORKING DIRECTORY, not the
# config file (unlike the data: paths) — the training script is normally launched from the
# repo root, while this notebook runs from downstream_apps/solar_event/, so check both.
candidate_dirs = [Path(cfg.output.ckpt_dir), Path("../..") / cfg.output.ckpt_dir]
ckpt_candidates = [p for d in candidate_dirs if d.is_dir() for p in d.glob("*.ckpt")]

CKPT_PATH = max(ckpt_candidates, key=lambda p: p.stat().st_mtime) if ckpt_candidates else None
print(f"Using checkpoint: {CKPT_PATH}")
# Override manually if this picked the wrong one, e.g.:
# CKPT_PATH = Path("/home/jovyan/surya_workshop/checkpoints/best-epoch=18-val_loss=0.6656.ckpt")

Loaded config for job: solar_event_identification
Using checkpoint: ../../checkpoints/best-epoch=00-val_loss=0.6750.ckpt


## Build the validation set

Reuses `build_datasets()` from the training script, so the channels, temporal sampling,
and flare-catalog alignment are identical to training — only `max_samples` differs (forced
to `None` above). The training loader is also returned but unused here.

In [3]:
ensure_assets(cfg, which=["scalers"] if TRAIN_BASELINE else ["scalers", "weights"])
scalers = build_scalers(info=cfg.data.scalers_path)

_, val_loader = build_datasets(cfg, scalers)
print(f"val: {len(val_loader.dataset)} samples | batch_size: {val_loader.batch_size}")

val: 96 samples | batch_size: 2


## Rebuild the model and load the trained weights

`build_model()` reconstructs the exact architecture the checkpoint was trained with
(backbone + LoRA adapters + head, wrapped in `SolarEventLightningModule`) from the same
config — including loading the *pretrained* backbone, which the checkpoint's weights then
overwrite. This only works if `cfg` here matches what the checkpoint was actually trained
with (LoRA/freeze/pooling settings in particular) — a mismatch shows up as a
`state_dict` key or shape error below, not a silent wrong answer.

Unlike `load_pretrained_weights()` (which loads a bare backbone state dict), a Lightning
checkpoint bundles optimizer/callback state alongside the tensors, so it's loaded with
`weights_only=False` rather than the `weights_only=True` used for the raw backbone
checkpoint.

In [4]:
assert CKPT_PATH is not None, "No checkpoint found — set CKPT_PATH manually."

lit_model = build_model(cfg, scalers, train_baseline=TRAIN_BASELINE)

checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lit_model.load_state_dict(checkpoint["state_dict"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lit_model = lit_model.to(device).eval()
print(f"Loaded {CKPT_PATH} onto {device}")

/home/jovyan/envs/surya_ws/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading pretrained weights from /home/jovyan/surya_workshop/downstream_apps/solar_event/assets/surya.366m.v1.pt.


Loaded 156 / 159 pretrained weights.
Applying PEFT LoRA: r=8, alpha=8, dropout=0.1, modules=['fc1', 'fc2', 'attn.qkv', 'attn.proj']


[LoRA] Adapted modules (36):
[LoRA]   backbone.backbone.blocks_attention.0.attn.proj
[LoRA]   backbone.backbone.blocks_attention.0.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.1.attn.proj
[LoRA]   backbone.backbone.blocks_attention.1.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.2.attn.proj
[LoRA]   backbone.backbone.blocks_attention.2.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.3.attn.proj
[LoRA]   backbone.backbone.blocks_attention.3.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.4.attn.proj
[LoRA]   backbone.backbone.blocks_atten

Loaded ../../checkpoints/best-epoch=00-val_loss=0.6750.ckpt onto cuda


## Run inference over the validation set

Mirrors `SolarEventLightningModule.validation_step`: apply `preprocess_fn` (only set for
the linear baseline) before the model call, then `sigmoid` the logits to get a probability
of class 1. `forward()` itself does not apply `preprocess_fn` — only the training/validation
steps do — so it has to be applied by hand here.

In [5]:
all_probs = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        target = batch["forecast"].reshape(-1).float()
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        if lit_model.preprocess_fn is not None:
            batch = lit_model.preprocess_fn(batch)

        logits = lit_model(batch)
        probs = logits.reshape(-1).sigmoid().cpu()

        all_probs.append(probs)
        all_targets.append(target)

y_prob = torch.cat(all_probs).numpy()
y_true = torch.cat(all_targets).numpy().astype(int)
print(f"Evaluated {len(y_true)} validation samples | positive rate: {y_true.mean():.3f}")

KeyboardInterrupt: 

## Confusion matrix and per-class metrics

`THRESHOLD` is the sigmoid-probability cutoff for predicting class 1 — lower it (e.g. to
0.3) to trade precision for recall on the positive class, independent of any change to
training (e.g. `pos_weight` in the loss). Do this after looking at the matrix below, not
before — the direction of any fix should follow from what the errors actually look like.

In [ ]:
THRESHOLD = 0.5
LABELS = ["No event", "Event"]

y_pred = (y_prob >= THRESHOLD).astype(int)
accuracy = (y_pred == y_true).mean()

print(f"Accuracy @ threshold={THRESHOLD}: {accuracy:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Solar event confusion matrix (threshold={THRESHOLD})")
plt.show()